# Exam Countdown Planner (Agentic AI) Demo

This notebook demonstrates the execution of the agentic plan-act loop of the **Exam Countdown Planner**. It runs in-process using FastAPI's `TestClient` to make execution simple and self-contained.

The demo covers three scenarios:
1. **Goal 1 (Cold Start)**: Setting the exam date and allocating topics in a single user message.
2. **Goal 2 (Missed Day)**: Simulating missing a day, triggering the catch-up shuffle logic forward using session memory.
3. **Goal 3 (Edge Case)**: Planning a very short study window (more topics than days) to test the round-robin packing compression.

In [ ]:
import os
import json

# Override 'today' to make output deterministic and independent of execution date
os.environ['TODAY_OVERRIDE'] = '2026-08-24'

from fastapi.testclient import TestClient
from app.main import app

# Initialize the test client
client = TestClient(app)

def run_demo_step(session_id: str, message: str):
    print(f"\033[1;34m[USER MESSAGE]\033[0m: '{message}'")
    response = client.post("/chat", json={"session_id": session_id, "message": message})
    assert response.status_code == 200, f"API failed: {response.text}"
    data = response.json()
    
    print("\n\033[1;35m--- AGENT PLAN-ACT LOOP TRACE ---\033[0m")
    for step in data["trace"]:
        step_type = step["type"]
        if step_type == "assistant_thought":
            print(f"\033[1;30m[Thought]\033[0m: {step['content']}")
        elif step_type == "tool_call":
            print(f"\033[1;33m[Tool Call]\033[0m: {step['tool']}({step['args']})")
        elif step_type == "tool_result":
            print(f"\033[1;32m[Tool Result]\033[0m: {step['tool']} returned: {step['result']}")
        elif step_type == "final_answer":
            print(f"\033[1;36m[Final Answer]\033[0m: {step['content']}")
        elif step_type == "error":
            print(f"\033[1;31m[Error]\033[0m: {step['content']}")
            
    print("\n\033[1;35m--- SESSION MEMORY STATE ---\033[0m")
    state = data["state"]
    print(f"Exam Date: {state.get('exam_date')}")
    print(f"Days Remaining: {state.get('days_left')}")
    print(f"Completed Days: {state.get('completed_days')}")
    print(f"Missed Days: {state.get('missed_days')}")
    print("Day-by-Day Study Plan:")
    plan = state.get("day_plan", {})
    try:
        sorted_days = sorted(plan.items(), key=lambda x: int(x[0].split()[1]))
        for day, topics in sorted_days:
            print(f"  {day}: {topics}")
    except Exception:
        for day, topics in plan.items():
            print(f"  {day}: {topics}")
    print("\n" + "="*75 + "\n")

## Goal 1: Cold Start (Multi-step Tool Calling)

Here we provide the exam date and study topics in a single message. The agent should reason that it must first set the exam date and then allocate the topics, executing both tools in sequence.

In [ ]:
# Ensure a clean session state
client.post("/reset", json={"session_id": "demo_session_1"})

run_demo_step(
    session_id="demo_session_1", 
    message="My exam is on 2026-09-15. I need to study Algebra, Chemistry, History, Biology, and Grammar."
)

## Goal 2: Missed Day & Catch-Up Shuffle

In the same session (`demo_session_1`), we report that we missed Day 3. The agent should recall the current exam date, topics, and schedule from memory, mark Day 3 as missed, and reshuffle all future topics forward.

In [ ]:
run_demo_step(
    session_id="demo_session_1", 
    message="I missed Day 3, I was sick. Can we update the plan?"
)

## Goal 3: Edge Case (Short Schedule / Pack Compression)

We create a new session (`demo_session_2`) with a very short window: the exam is in 2 days, and we have 6 topics to study. The round-robin allocation should pack multiple topics on the same days so that everything is covered before the exam.

In [ ]:
# Ensure a clean session state
client.post("/reset", json={"session_id": "demo_session_2"})

run_demo_step(
    session_id="demo_session_2", 
    message="My exam is on 2026-08-26. I need to study Algebra, Chemistry, History, Biology, Grammar, and Literature."
)